[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/python_tracing_quickstart.ipynb)


# Phoenix 追踪教程：客服智能体（Python）

本 Notebook 配合 Phoenix 追踪教程文档使用：
- [第 1 章：首次追踪](/docs/phoenix/tracing/tutorial/your-first-traces)
- [第 2 章：标注与评估](/docs/phoenix/tracing/tutorial/annotations-and-evaluations)
- [第 3 章：会话](/docs/phoenix/tracing/tutorial/sessions)


# 第 1 章：首次追踪

## 配置追踪

### 安装依赖


In [ ]:
!pip install arize-phoenix arize-phoenix-otel arize-phoenix-client openai openinference-instrumentation-openai openinference-instrumentation numpy

In [ ]:
import os

# 指向 Phoenix 实例。本地 `uvx arize-phoenix serve` 监听
# http://localhost:6006，无需 API 密钥。如果 Phoenix 部署
# 在其他位置，请改用相应端点。
os.environ["OPENAI_API_KEY"] = "your-openai-api-key"
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "http://localhost:6006"

# 仅在 Phoenix 实例启用身份验证时需要：
# os.environ["PHOENIX_API_KEY"] = "your-phoenix-api-key"

# 若端点没有服务监听（例如 Colab 等托管环境中，
# localhost 没有 Phoenix 服务），就在本 Notebook 中启动 Phoenix。
from urllib.request import urlopen

try:
    urlopen(os.environ["PHOENIX_COLLECTOR_ENDPOINT"], timeout=5)
except Exception:
    import phoenix as px

    px.launch_app()

### 配置追踪


In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="support-bot", auto_instrument=True)

## 追踪 LLM 调用

导入库并设置 OpenAI 客户端。


In [ ]:
import json
import re
import uuid
from typing import Any, Dict, List, Literal, Optional, TypedDict

from openai import OpenAI
from opentelemetry import trace
from phoenix.client import Client

client = OpenAI()
phoenix_client = Client()

# 获取用于创建自定义跨度的追踪器
tracer = trace.get_tracer("support-agent")

这是一个启用追踪的简单 LLM 调用。所有 OpenAI 调用都会自动追踪。

In [ ]:
user_query = "Where is my order?"

result = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "将查询分类为 'order_status' 或 'faq'"},
        {"role": "user", "content": user_query},
    ],
)
print("\n✅ 此 LLM 调用已自动追踪！在 Phoenix 界面查看跨度。")

## 追踪工具调用

设置客服智能体可访问的模拟数据。


In [ ]:
class Message(TypedDict):
    role: Literal["user", "assistant"]
    content: str


# 订单数据库（供工具调用）
ORDER_DATABASE: Dict[str, Dict[str, str]] = {
    "ORD-12345": {
        "status": "shipped",
        "carrier": "FedEx",
        "trackingNumber": "1234567890",
        "eta": "December 11, 2025",
    },
    "ORD-67890": {
        "status": "processing",
        "carrier": "pending",
        "trackingNumber": "pending",
        "eta": "December 15, 2025",
    },
    "ORD-11111": {
        "status": "delivered",
        "carrier": "UPS",
        "trackingNumber": "9876543210",
        "eta": "Delivered December 5, 2025",
    },
}


# 常见问答数据库（供 RAG 使用）
class FAQEntry(TypedDict):
    id: int
    question: str
    answer: str
    category: str
    embedding: Optional[List[float]]


FAQ_DATABASE: List[FAQEntry] = [
    {
        "id": 1,
        "question": "How do I reset my password?",
        "answer": "Go to Settings > Security > Reset Password. You'll receive an email with a reset link that expires in 24 hours.",
        "category": "Account",
        "embedding": None,
    },
    {
        "id": 2,
        "question": "What's your refund policy?",
        "answer": "We offer full refunds within 30 days of purchase for unused items. Contact support with your order number to initiate a refund.",
        "category": "Billing",
        "embedding": None,
    },
    {
        "id": 3,
        "question": "How do I cancel my subscription?",
        "answer": "Go to Account Settings > Subscription > Cancel Subscription. Your access continues until the end of the current billing period.",
        "category": "Billing",
        "embedding": None,
    },
    {
        "id": 4,
        "question": "What payment methods do you accept?",
        "answer": "We accept Visa, Mastercard, American Express, PayPal, and Apple Pay. All transactions are securely processed.",
        "category": "Billing",
        "embedding": None,
    },
    {
        "id": 5,
        "question": "How do I update my profile information?",
        "answer": "Go to Account Settings > Profile. You can update your name, email, phone number, and address there.",
        "category": "Account",
        "embedding": None,
    },
]

QueryCategory = Literal["order_status", "faq"]


class ClassificationResult(TypedDict):
    category: QueryCategory
    confidence: str
    reasoning: str


class AgentResponse(TypedDict):
    query: str
    response: str
    spanId: str
    category: QueryCategory
    sessionId: Optional[str]


class SessionContext(TypedDict):
    lastMentionedOrderId: Optional[str]
    turnCount: int

使用工具调用的示例查询。工具让智能体能够与数据库、API 和外部系统交互。

In [ ]:
from openinference.semconv.trace import SpanAttributes

tools = [
    {
        "type": "function",
        "function": {
            "name": "lookupOrderStatus",
            "description": "根据订单 ID 查询客户订单当前状态",
            "parameters": {
                "type": "object",
                "properties": {
                    "orderId": {
                        "type": "string",
                        "description": "要查询的订单 ID，例如 ORD-12345",
                    }
                },
                "required": ["orderId"],
            },
        },
    }
]


# 自动执行工具的辅助函数
def execute_tool_call(tool_call, database):
    """执行工具调用并返回结果。"""
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)

    with tracer.start_as_current_span(
        function_name,
        attributes={
            SpanAttributes.OPENINFERENCE_SPAN_KIND: "TOOL",
            SpanAttributes.TOOL_NAME: function_name,
            SpanAttributes.TOOL_PARAMETERS: json.dumps(function_args),
            SpanAttributes.INPUT_VALUE: json.dumps(function_args),
        },
    ) as tool_span:
        if function_name == "lookupOrderStatus":
            order_id = function_args.get("orderId")
            result = database.get(order_id, {"error": f"Order {order_id} not found"})
        else:
            result = {"error": f"未知工具： {function_name}"}

        tool_span.set_attribute(SpanAttributes.OUTPUT_VALUE, json.dumps(result))
        tool_span.set_status(trace.Status(trace.StatusCode.OK))
        return result


user_query = "What is the status of ORD-12345?"

# 创建父跨度以组织所有跨度
with tracer.start_as_current_span(
    "tool-call-example",
    attributes={
        SpanAttributes.OPENINFERENCE_SPAN_KIND: "CHAIN",
        SpanAttributes.INPUT_VALUE: user_query,
    },
) as parent_span:
    messages = [
        {
            "role": "system",
            "content": "你是乐于助人的客服智能体。客户询问订单状态时，使用 lookupOrderStatus 工具获取信息。",
        },
        {"role": "user", "content": user_query},
    ]

    result = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )

    message = result.choices[0].message
    messages.append(message)

    # 若调用了工具，执行工具后获取最终回复
    if message.tool_calls:
        for tool_call in message.tool_calls:
            tool_result = execute_tool_call(tool_call, ORDER_DATABASE)
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result),
                }
            )

        # 携带工具结果发起最终 LLM 调用
        final_result = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
        )
        final_response = final_result.choices[0].message.content
    else:
        final_response = message.content

    parent_span.set_attribute(SpanAttributes.OUTPUT_VALUE, final_response)
    parent_span.set_status(trace.Status(trace.StatusCode.OK))
    print(f"查询： {user_query}")
print(f"回复： {final_response}")
print("✅ 在 Phoenix 界面查看完整追踪")

## 追踪 RAG 流水线

用于嵌入和相似度搜索的辅助函数。


In [ ]:
import numpy as np


def cosine_similarity(a: List[float], b: List[float]) -> float:
    """计算两个向量之间的余弦相似度。"""
    a_array = np.array(a)
    b_array = np.array(b)
    dot_product = np.dot(a_array, b_array)
    magnitude_a = np.linalg.norm(a_array)
    magnitude_b = np.linalg.norm(b_array)
    return dot_product / (magnitude_a * magnitude_b)


def initialize_faq_embeddings() -> None:
    print("📚 正在初始化常见问答嵌入……")

    for faq in FAQ_DATABASE:
        response = client.embeddings.create(model="text-embedding-ada-002", input=faq["question"])
        faq["embedding"] = response.data[0].embedding

    print("✅ 常见问答嵌入已初始化")

启用追踪的 RAG 流水线示例：

In [ ]:
# 先初始化嵌入（只需执行一次）
initialize_faq_embeddings()

In [ ]:
user_query = "How do I reset my password?"

with tracer.start_as_current_span(
    "rag-example",
    attributes={
        SpanAttributes.OPENINFERENCE_SPAN_KIND: "CHAIN",
        SpanAttributes.INPUT_VALUE: user_query,
    },
) as parent_span:
    # 第 1 步：嵌入查询（自动追踪）
    embedding_response = client.embeddings.create(model="text-embedding-ada-002", input=user_query)
    query_embedding = embedding_response.data[0].embedding

    # 第 2 步：通过余弦相似度查找相关常见问答
    faq_scores = []
    for faq in FAQ_DATABASE:
        if faq["embedding"]:
            score = cosine_similarity(query_embedding, faq["embedding"])
            faq_scores.append((faq, score))

    relevant_faqs = sorted(faq_scores, key=lambda x: x[1], reverse=True)[:2]

    with tracer.start_as_current_span(
        "faq-retrieval",
        attributes={
            SpanAttributes.OPENINFERENCE_SPAN_KIND: "RETRIEVER",
            SpanAttributes.INPUT_VALUE: user_query,
        },
    ) as retrieval_span:
        for i, (faq, score) in enumerate(relevant_faqs):
            retrieval_span.set_attribute(f"retrieval.documents.{i}.document.id", str(faq["id"]))
            retrieval_span.set_attribute(
                f"retrieval.documents.{i}.document.content",
                f"Q: {faq['question']}\nA: {faq['answer']}",
            )
            retrieval_span.set_attribute(
                f"retrieval.documents.{i}.document.metadata",
                json.dumps({"category": faq["category"], "score": score}),
            )

        retrieval_span.set_status(trace.Status(trace.StatusCode.OK))

    # 第 3 步：根据检索到的常见问答构建上下文
    rag_context = "\n\n".join(
        [f"Q: {faq['question']}\nA: {faq['answer']}" for faq, _ in relevant_faqs]
    )

    # 第 4 步：根据检索上下文生成答案（自动追踪）
    rag_result = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": f"你是乐于助人的客服智能体。仅使用以下上下文中的信息回答用户问题，语气友好且简洁。\n\n上下文：\n{rag_context}",
            },
            {"role": "user", "content": user_query},
        ],
    )

    final_response = rag_result.choices[0].message.content
    parent_span.set_attribute(SpanAttributes.OUTPUT_VALUE, final_response)
    parent_span.set_status(trace.Status(trace.StatusCode.OK))
    print(f"查询： {user_query}")
print(f"回复： {final_response}")
print("\n✅ 全部 RAG 操作均已追踪！")

## 使用父 span 对操作分组

完整的客服智能体，将所有操作包裹在一个父 span 中。


In [ ]:
from openinference.instrumentation import using_session
from opentelemetry.trace import format_span_id


def handle_support_query(
    user_query: str,
    session_id: Optional[str] = None,
    conversation_history: List[Message] = None,
    session_context: SessionContext = None,
) -> AgentResponse:
    """
    处理支持查询，并可选择跟踪会话。

    Args:
        user_query: 用户问题
        session_id: 多轮对话的可选会话 ID
        conversation_history: 对话中的先前消息
        session_context: 先前轮次的上下文
    """
    if conversation_history is None:
        conversation_history = []
    if session_context is None:
        session_context = {"lastMentionedOrderId": None, "turnCount": 0}

    def run_agent() -> AgentResponse:
        with tracer.start_as_current_span(
            "support-agent",
            attributes={
                SpanAttributes.OPENINFERENCE_SPAN_KIND: "AGENT",
                SpanAttributes.INPUT_VALUE: user_query,
                **({SpanAttributes.SESSION_ID: session_id} if session_id else {}),
                "conversation.turn": session_context["turnCount"] + 1,
            },
        ) as agent_span:
            # 记录跨度 ID
            span_id = format_span_id(agent_span.get_span_context().span_id)
            category: QueryCategory = "faq"

            try:
                print("\n" + "=" * 60)
                print("🤖 客服智能体正在处理查询")
                print("=" * 60)
                print(f'📨 查询： "{user_query}"')
                print(f"   跨度 ID： {span_id}")
                if session_id:
                    print(f"   会话 ID： {session_id}")
                    print(f"   轮次： {session_context['turnCount'] + 1}")

                # 为多轮客服构建对话上下文
                conversation_context = (
                    "\n\n先前对话：\n"
                    + "\n".join([f"{m['role']}: {m['content']}" for m in conversation_history])
                    if conversation_history
                    else ""
                )

                # 检查是否记住此前轮次的订单 ID
                remembered_order_info = (
                    f"\n注意：客户此前提到订单 {session_context['lastMentionedOrderId']}."
                    if session_context["lastMentionedOrderId"]
                    else ""
                )

                # 第 1 步：对查询分类
                print("\n📋 第 1 步：正在对查询分类……")

                classification_prompt = f"""你是支持查询分类器。请将用户查询划分为以下类别之一：

1. "order_status"：关于订单跟踪、配送状态、物流、订单位置、运单号和预计到达时间的问题
2. "faq"：关于账号、账单、退款、密码、订阅和支付方式的一般问题
{remembered_order_info}
仅回复 JSON：
{{
  "category": "order_status" or "faq",
  "confidence": "high" or "medium" or "low",
  "reasoning": "简要说明"
}}"""

                classification_response = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[
                        {"role": "system", "content": classification_prompt},
                        {"role": "user", "content": user_query + conversation_context},
                    ],
                    response_format={"type": "json_object"},
                )

                classification_text = classification_response.choices[0].message.content

                try:
                    classification: ClassificationResult = json.loads(classification_text)
                except json.JSONDecodeError:
                    # 解析失败时默认使用 FAQ
                    classification = {
                        "category": "faq",
                        "confidence": "low",
                        "reasoning": "分类结果解析失败",
                    }

                print(f"   类别： {classification['category']}")
                print(f"   理由： {classification['reasoning']}")

                category = classification["category"]
                agent_span.set_attribute("classification.category", classification["category"])
                agent_span.set_attribute("classification.confidence", classification["confidence"])

                # 第 2 步：根据分类路由
                if classification["category"] == "order_status":
                    print("\n🔧 第 2 步：正在决定是否使用工具……")

                    order_prompt = (
                        f'{user_query}\n\n注意：在此前对话中，客户提到订单 {session_context["lastMentionedOrderId"]}。若用户提到“那个订单”等类似表达，请使用此订单 ID。'
                        if session_context["lastMentionedOrderId"]
                        else user_query
                    )

                    # 定义订单查询工具
                    tools = [
                        {
                            "type": "function",
                            "function": {
                                "name": "lookupOrderStatus",
                                "description": "根据订单 ID 查询客户订单当前状态",
                                "parameters": {
                                    "type": "object",
                                    "properties": {
                                        "orderId": {
                                            "type": "string",
                                            "description": "要查询的订单 ID，例如 ORD-12345",
                                        }
                                    },
                                    "required": ["orderId"],
                                },
                            },
                        }
                    ]

                    tool_decision = client.chat.completions.create(
                        model="gpt-4o-mini",
                        messages=[
                            {
                                "role": "system",
                                "content": "你是乐于助人的客服智能体。客户询问订单状态时，使用 lookupOrderStatus 工具获取信息。 若当前及此前均未提及订单 ID，请礼貌索取。只要提供或引用了订单 ID，就必须使用工具。",
                            },
                            {"role": "user", "content": order_prompt},
                        ],
                        tools=tools,
                        tool_choice="auto",
                    )

                    # 检查是否调用工具
                    message = tool_decision.choices[0].message
                    order_info: Optional[Dict[str, Any]] = None

                    if message.tool_calls:
                        tool_call = message.tool_calls[0]
                        function_name = tool_call.function.name
                        function_args = json.loads(tool_call.function.arguments)
                        order_id = function_args.get("orderId")

                        print(f"   🔧 已调用工具： {function_name}({order_id})")

                        # 为工具调用创建跨度
                        with tracer.start_as_current_span(
                            function_name,
                            attributes={
                                SpanAttributes.OPENINFERENCE_SPAN_KIND: "TOOL",
                                SpanAttributes.TOOL_NAME: function_name,
                                SpanAttributes.TOOL_PARAMETERS: json.dumps(function_args),
                                SpanAttributes.INPUT_VALUE: json.dumps(function_args),
                            },
                        ) as tool_span:
                            order = ORDER_DATABASE.get(order_id)
                            if not order:
                                order_info = {"error": f"Order {order_id} not found in our system"}
                                tool_span.set_attribute(
                                    SpanAttributes.OUTPUT_VALUE, json.dumps(order_info)
                                )
                                tool_span.set_status(
                                    trace.Status(trace.StatusCode.ERROR, "Order not found")
                                )
                            else:
                                print(f"   ✅ 已找到订单： {json.dumps(order)}")
                                order_info = {"orderId": order_id, **order}
                                tool_span.set_attribute(
                                    SpanAttributes.OUTPUT_VALUE, json.dumps(order_info)
                                )
                                tool_span.set_status(trace.Status(trace.StatusCode.OK))

                    if order_info and "error" not in order_info:
                        print(f"   📦 回复使用的订单信息： {json.dumps(order_info)}")
                        print("\n💬 第 3 步：正在根据工具结果生成回复……")

                        final_response = client.chat.completions.create(
                            model="gpt-4o-mini",
                            messages=[
                                {
                                    "role": "system",
                                    "content": "你是乐于助人的客服智能体。请友好地总结订单信息，严格使用提供的数据，不得编造。",
                                },
                                {
                                    "role": "user",
                                    "content": f""""客户询问： "{user_query}"

以下是查到的订单信息：
- 订单 ID： {order_info["orderId"]}
- Status: {order_info["status"]}
- Carrier: {order_info["carrier"]}
- 运单号： {order_info["trackingNumber"]}
- 预计到达时间： {order_info["eta"]}

用友好的 2–3 句话向客户介绍这些信息。""",
                                },
                            ],
                        )

                        response = final_response.choices[0].message.content
                    else:
                        response = (
                            message.content
                            or "很乐意帮助查询订单状态。请提供订单 ID，格式类似 ORD-XXXXX。"
                        )

                else:
                    # 使用 RAG 处理常见问答
                    print("\n📚 第 2 步：正在搜索知识库（RAG）……")

                    # 嵌入查询
                    embedding_response = client.embeddings.create(
                        model="text-embedding-ada-002", input=user_query
                    )
                    query_embedding = embedding_response.data[0].embedding

                    # 查找相关常见问答
                    faq_scores = []
                    for faq in FAQ_DATABASE:
                        if faq["embedding"]:
                            score = cosine_similarity(query_embedding, faq["embedding"])
                            faq_scores.append((faq, score))

                    relevant_faqs = sorted(faq_scores, key=lambda x: x[1], reverse=True)[:2]

                    print("   已找到相关常见问答")

                    # 创建检索跨度以跟踪检索操作
                    with tracer.start_as_current_span(
                        "faq-retrieval",
                        attributes={
                            SpanAttributes.OPENINFERENCE_SPAN_KIND: "RETRIEVER",
                            SpanAttributes.INPUT_VALUE: user_query,
                        },
                    ) as retrieval_span:
                        for i, (faq, score) in enumerate(relevant_faqs):
                            retrieval_span.set_attribute(
                                f"retrieval.documents.{i}.document.id", str(faq["id"])
                            )
                            retrieval_span.set_attribute(
                                f"retrieval.documents.{i}.document.content",
                                f"Q: {faq['question']}\nA: {faq['answer']}",
                            )
                            metadata_str = json.dumps(
                                {"category": faq["category"], "score": float(score)}
                            )
                            retrieval_span.set_attribute(
                                f"retrieval.documents.{i}.document.metadata", metadata_str
                            )

                        retrieval_span.set_status(trace.Status(trace.StatusCode.OK))

                    # 构建上下文
                    rag_context = "\n\n".join(
                        [f"Q: {faq['question']}\nA: {faq['answer']}" for faq, _ in relevant_faqs]
                    )

                    # 生成答案
                    print("\n💬 第 3 步：正在生成回复……")

                    rag_result = client.chat.completions.create(
                        model="gpt-4o-mini",
                        messages=[
                            {
                                "role": "system",
                                "content": f"你是乐于助人的客服智能体。仅使用以下上下文中的信息回答用户问题，语气友好且简洁。\n\n上下文：\n{rag_context}",
                            },
                            {"role": "user", "content": user_query},
                        ],
                    )

                    response = rag_result.choices[0].message.content

                print(f"\n📤 回复： {response}")
                print("=" * 60)

                agent_span.set_attribute(SpanAttributes.OUTPUT_VALUE, response)
                agent_span.set_status(trace.Status(trace.StatusCode.OK))

                return {
                    "query": user_query,
                    "response": response,
                    "spanId": span_id,
                    "category": category,
                    "sessionId": session_id,
                }
            except Exception as error:
                agent_span.set_status(trace.Status(trace.StatusCode.ERROR, str(error)))
                raise

    # 若有会话 ID，则传递给所有子跨度
    if session_id:
        with using_session(session_id):
            return run_agent()

    return run_agent()

In [ ]:
# 先确保常见问答嵌入已初始化
initialize_faq_embeddings()

queries = [
    "What's the status of order ORD-12345?",
    "How can I get a refund?",
    "Where is my order ORD-67890?",
    "I forgot my password",
    "What's the status of order ORD-99999?",
    "How do I upgrade to premium?",
    "Can you help me with something?",
]

print("=" * 60)
print("使用测试查询运行客服智能体")
print("=" * 60)

for query in queries:
    result = handle_support_query(query)
    print(f"\n✅ 已处理查询： {result['category']}")
    print("-" * 60)

print("\n✅ 全部查询处理完毕！在 Phoenix 界面查看追踪。")

# 第 2 章：标注与评估

## 通过代码添加标注（用户反馈）

### 从运行中的代码获取 span ID

客服智能体已经捕获了 span ID。现在收集用户反馈，并将其记录到 Phoenix。


In [ ]:
from phoenix.client.resources.spans import SpanAnnotationData


def collect_user_feedback(responses: List[AgentResponse]) -> None:
    """
    针对每条回复收集用户交互反馈。
    显示查询与回复，然后请用户点赞或点踩。
    """
    print("\n" + "=" * 60)
    print("👍👎 用户反馈收集")
    print("=" * 60)
    print("\n针对每条回复，请输入：")
    print("  'y' 或 '1' = 👍 点赞（回复良好）")
    print("  'n' 或 '0' = 👎 点踩（回复不佳）")
    print("  's' = 跳过（不提供反馈）")
    print("")

    annotations: List[SpanAnnotationData] = []

    for i, resp in enumerate(responses):
        print(f"回复 {i + 1} of {len(responses)}")
        print(f'查询： "{resp["query"]}"')
        print(f'回复： "{resp["response"]}"')

        answer = input("这条回复有帮助吗？(y/n/s)： ").strip().lower()

        if answer in ["y", "1", "yes"]:
            print("   → 👍 已记录为点赞\n")
            annotations.append(
                SpanAnnotationData(
                    name="user_feedback",
                    span_id=resp["spanId"],
                    annotator_kind="HUMAN",
                    result={"label": "thumbs-up", "score": 1.0},
                    metadata={"category": resp["category"], "source": "interactive_tutorial"},
                )
            )
        elif answer in ["n", "0", "no"]:
            print("   → 👎 已记录为点踩\n")
            annotations.append(
                SpanAnnotationData(
                    name="user_feedback",
                    span_id=resp["spanId"],
                    annotator_kind="HUMAN",
                    result={"label": "thumbs-down", "score": 0.0},
                    metadata={"category": resp["category"], "source": "interactive_tutorial"},
                )
            )
        else:
            print("   → ⏭️  已跳过\n")

    if annotations:
        print("-" * 60)

        try:
            phoenix_client.spans.log_span_annotations(
                span_annotations=annotations,
                sync=False,
            )
            print(f"✅ 已记录 {len(annotations)} 条反馈标注到 Phoenix")
        except Exception as error:
            print(f"记录反馈失败： {error}")

示例：收集针对客服智能体回答的反馈。


In [ ]:
queries = [
    "What's the status of order ORD-12345?",
    "How can I get a refund?",
    "I forgot my password",
]

print("=" * 60)
print("运行客服智能体查询")
print("=" * 60)

responses = []
for query in queries:
    result = handle_support_query(query)
    responses.append(result)

collect_user_feedback(responses)

## 使用 LLM 作为裁判进行评估

### 安装 Phoenix Evals 包

In [ ]:
!pip install arize-phoenix-evals

### 工具结果评估器

工具调用成功了还是返回了错误？下面用简单的代码检查：

In [ ]:
# 从 Phoenix 获取跨度
spans = phoenix_client.spans.get_spans(
    project_identifier="support-bot",
    limit=200,
)

# 筛选工具跨度（lookupOrderStatus）
tool_spans = [span for span in spans if span.get("name") == "lookupOrderStatus"]

print(f"已找到 {len(tool_spans)} 个工具跨度")

# 工具结果评估器：基于代码的检查
tool_annotations = []

for span in tool_spans:
    # 从 context 获取 span_id
    context = span.get("context", {})
    span_id = context.get("span_id", "") if isinstance(context, dict) else ""

    # 获取属性（可能为字典或 JSON 字符串）
    attributes = span.get("attributes", {})
    if isinstance(attributes, str):
        attributes = json.loads(attributes)

    output_value = attributes.get("output.value", "")

    # 简单检查：输出是否包含 "error" 或 "not found"？
    output_str = json.dumps(output_value) if not isinstance(output_value, str) else output_value
    has_error = "error" in output_str.lower() or "not found" in output_str.lower()

    status = "❌ ERROR" if has_error else "✅ SUCCESS"

    tool_annotations.append(
        SpanAnnotationData(
            name="tool_result",
            span_id=span_id,
            annotator_kind="CODE",
            result={
                "label": "error" if has_error else "success",
                "score": 0.0 if has_error else 1.0,
            },
            metadata={
                "evaluator": "tool_result",
                "type": "code",
            },
        )
    )

print(f"\n✅ 已评估 {len(tool_annotations)} 个工具跨度")

### 检索相关性评估器

检索到的上下文是否与问题相关？

In [ ]:
from phoenix.evals import LLM, ClassificationEvaluator

# 筛选检索跨度（RETRIEVER 类型）：常见问答检索
retrieval_spans = [
    span
    for span in spans
    if span.get("span_kind") == "RETRIEVER" or span.get("name") == "faq-retrieval"
]

print(f"已找到 {len(retrieval_spans)} 个常见问答检索跨度")

# 创建 LLM 裁判评估器，判断检索上下文是否相关
llm = LLM(provider="openai", model="gpt-5")

retrieval_relevance_evaluator = ClassificationEvaluator(
    name="retrieval_relevance",
    prompt_template="""你正在评估检索上下文是否与回答用户提示词相关。

将检索分类为：
- RELEVANT：上下文包含直接帮助回答问题的信息
- IRRELEVANT：上下文不含对该问题有用的信息

你需要比较 "Context" 对象与 "prompt" 对象。

[上下文与提示词]：{input}""",
    llm=llm,
    choices={"relevant": 1, "irrelevant": 0},
)

# 评估每个检索跨度
rag_annotations = []

for span in retrieval_spans:
    # 从 context 获取 span_id
    context = span.get("context", {})
    span_id = context.get("span_id", "") if isinstance(context, dict) else ""

    # 获取属性（可能为字典或 JSON 字符串）
    attributes = span.get("attributes", {})
    if isinstance(attributes, str):
        attributes = json.loads(attributes)

    # 提取查询与检索文档
    query = attributes.get("input.value", "")

    # 提取检索文档
    documents = []
    i = 0
    while f"retrieval.documents.{i}.document.content" in attributes:
        doc_content = attributes.get(f"retrieval.documents.{i}.document.content", "")
        documents.append(doc_content)
        i += 1

    if not query or not documents:
        span_id_short = span_id[:8] if span_id else "unknown"
        print(f"   跳过跨度 {span_id_short}：缺少查询或文档")
        continue

    # 构建评估器输入：查询与检索上下文
    context_text = "\n\n".join(documents)
    evaluation_input = f"查询： {query}\n\n检索上下文：\n{context_text}"

    try:
        result = retrieval_relevance_evaluator.evaluate({"input": evaluation_input})
        score_result = result[0] if isinstance(result, list) else result

        status = "✅ RELEVANT" if score_result.label == "relevant" else "❌ IRRELEVANT"
        span_id_short = span_id[:8] if span_id else "unknown"
        print(f"   检索跨度 {span_id_short}... {status}")

        rag_annotations.append(
            SpanAnnotationData(
                name="retrieval_relevance",
                span_id=span_id,
                annotator_kind="LLM",
                result={
                    "label": score_result.label,
                    "score": score_result.score
                    if hasattr(score_result, "score")
                    else (1.0 if score_result.label == "relevant" else 0.0),
                },
                metadata={
                    "model": "gpt-5",
                    "evaluator": "retrieval_relevance",
                },
            )
        )
    except Exception as e:
        span_id_short = span_id[:8] if span_id else "unknown"
        print(f"   评估跨度出错： {span_id_short}: {e}")

print(f"\n✅ 已评估 {len(rag_annotations)} 个检索跨度")

### 将评估结果记录到 Phoenix

将所有评估标注记录到 Phoenix：

In [ ]:
all_eval_annotations = tool_annotations + rag_annotations

if all_eval_annotations:
    print(f"\n📤 正在记录 {len(all_eval_annotations)} 条评估标注到 Phoenix……")

    try:
        phoenix_client.spans.log_span_annotations(
            span_annotations=all_eval_annotations,
            sync=False,
        )
        print(f"✅ 已记录 {len(all_eval_annotations)} 条评估标注")
        print(f"   - {len(tool_annotations)} 条 tool_result 标注")
        print(f"   - {len(rag_annotations)} 条 retrieval_relevance 标注")
    except Exception as error:
        print(f"❌ 记录评估失败： {error}")
else:
    print("\n⚠️  没有可记录的标注。请先运行客服智能体。")

# 第 3 章：会话

## 配置会话

### 为智能体添加会话追踪

客服智能体已经支持会话。下面演示如何运行多轮对话。


In [ ]:
class ConversationTurn(TypedDict):
    userMessage: str
    expectedBehavior: str


class ConversationScenario(TypedDict):
    name: str
    description: str
    turns: List[ConversationTurn]


def run_multi_turn_conversation(scenario: ConversationScenario) -> Dict[str, Any]:
    """
    运行带会话追踪的多轮对话。
    每段对话获得唯一会话 ID，全部轮次彼此关联。
    """
    session_id = str(uuid.uuid4())
    responses: List[AgentResponse] = []
    conversation_history: List[Message] = []
    session_context: SessionContext = {"lastMentionedOrderId": None, "turnCount": 0}

    print("\n" + "=" * 60)
    print(f"🗣️  对话： {scenario['name']}")
    print(f"📝 {scenario['description']}")
    print(f"🔑 会话 ID： {session_id}")
    print("=" * 60)

    for turn in scenario["turns"]:
        print(f'\n💬 第几轮： {session_context["turnCount"] + 1}: "{turn["userMessage"]}"')
        print(f"   预期： {turn['expectedBehavior']}")

        # 携带会话上下文运行智能体
        result = handle_support_query(
            turn["userMessage"], session_id, conversation_history, session_context
        )

        responses.append(result)

        # 为下一轮更新对话历史
        conversation_history.append({"role": "user", "content": turn["userMessage"]})
        conversation_history.append({"role": "assistant", "content": result["response"]})

        # 更新会话上下文：提取提及的订单 ID
        order_id_match = re.search(r"ORD-\d+", turn["userMessage"], re.IGNORECASE)
        if not order_id_match:
            order_id_match = re.search(r"ORD-\d+", result["response"], re.IGNORECASE)
        if order_id_match:
            session_context["lastMentionedOrderId"] = order_id_match.group(0).upper()
        session_context["turnCount"] += 1

    print("\n" + "-" * 60)
    print(f"✅ 对话完成： {len(scenario['turns'])} 轮")
    print("-" * 60)

    return {"sessionId": session_id, "responses": responses}


def run_sessions_demo() -> None:
    """使用多个对话场景运行多轮会话演示。"""
    print("=" * 60)
    print("Phoenix 追踪教程：会话演示")
    print("=" * 60)
    print("\n本演示展示如何将多轮对话作为会话追踪。")
    print("每段对话有唯一会话 ID，将全部轮次关联起来。")
    print("在 Phoenix 界面的 'Sessions' 标签页中查看。\n")

    # 先初始化常见问答嵌入
    initialize_faq_embeddings()

    # 定义对话场景
    scenarios: List[ConversationScenario] = [
        {
            "name": "订单查询：成功解决",
            "description": "客户查询订单、获取状态并继续追问",
            "turns": [
                {
                    "userMessage": "What's the status of order ORD-12345?",
                    "expectedBehavior": "工具调用 → 返回已发货状态",
                },
                {
                    "userMessage": "When will it arrive?",
                    "expectedBehavior": "智能体记住订单 → 根据先前查询提供预计到达时间",
                },
                {
                    "userMessage": "What's the tracking number?",
                    "expectedBehavior": "智能体记住订单 → 提供运单号",
                },
            ],
        },
        {
            "name": "常见问答对话",
            "description": "客户在同一会话提出多个常见问题",
            "turns": [
                {
                    "userMessage": "How do I reset my password?",
                    "expectedBehavior": "RAG → 密码重置说明",
                },
                {
                    "userMessage": "And what about refunds?",
                    "expectedBehavior": "RAG → 退款政策信息",
                },
            ],
        },
        {
            "name": "混合对话：上下文测试",
            "description": "客户切换订单与常见问答主题",
            "turns": [
                {
                    "userMessage": "Check my order ORD-67890",
                    "expectedBehavior": "工具调用 → 处理中状态",
                },
                {
                    "userMessage": "How do I cancel my subscription?",
                    "expectedBehavior": "RAG → 取消订阅说明（不同主题）",
                },
                {
                    "userMessage": "Back to my order - what's the carrier?",
                    "expectedBehavior": "智能体应记住此前提到的 ORD-67890",
                },
            ],
        },
    ]

    # 运行所有对话场景
    all_responses: List[AgentResponse] = []
    session_ids: List[str] = []

    for scenario in scenarios:
        result = run_multi_turn_conversation(scenario)
        all_responses.extend(result["responses"])
        session_ids.append(result["sessionId"])

    # 刷新追踪
    print("\n⏳ 正在刷新追踪……")
    tracer_provider.force_flush()
    print("✅ 追踪已刷新！")

    # 收集反馈
    collect_user_feedback(all_responses)
    # 汇总
    print("\n" + "=" * 60)
    print("📊 会话汇总")
    print("=" * 60)
    print(f"\n   对话数： {len(scenarios)}")
    print(f"   总轮次： {len(all_responses)}")
    print("\n   会话 ID： ")
    for i, session_id in enumerate(session_ids):
        print(f"   {i + 1}. {session_id} ({scenarios[i]['name']})")
    print("\n" + "=" * 60)
    print("## 查看与分析会话")
    print("=" * 60)
    print("\n查看要点：")
    print("   1. 在项目中点击 'Sessions' 标签页")
    print("   2. 每段对话会显示为独立会话")
    print("   3. 点击会话，查看类似聊天界面的历史记录")
    print("   4. 注意所有轮次如何共享同一个会话 ID")
    print("   5. 查看整个对话的令牌用量与延迟")
    print("=" * 60)

## 运行多轮对话

运行多轮会话示例，查看如何将对话作为连贯的整体追踪。


In [ ]:
run_sessions_demo()

### 会话级评估

使用 LLM 裁判自动评估整段对话，可以减少逐个手动审阅会话的工作。这有助于回答：智能体是否记住了上下文？问题是否得到解决？对话在哪一步出现了问题？

#### 对话连贯性评估器

此评估器检查智能体是否在整段对话中保持上下文：

In [ ]:
from phoenix.evals import LLM, ClassificationEvaluator

llm = LLM(provider="openai", model="gpt-5")

conversation_coherence_evaluator = ClassificationEvaluator(
    name="conversation_coherence",
    prompt_template="""你正在评估客服智能体是否在多轮对话中保持上下文。

对话满足以下条件时为 COHERENT：
- 智能体记得之前轮次的信息
- 不要求用户重复已经提供的信息
- 回复恰当承接先前上下文
- 对话自然流畅

以下情况为 INCOHERENT：
- 智能体“忘记”客户先前说过的内容
- 多次索取相同信息
- 回复与先前轮次脱节
- 客户必须重复表达

[完整对话]：
{input}

智能体是否在整个对话中保持了上下文？""",
    llm=llm,
    choices={"coherent": 1, "incoherent": 0},
)

#### 问题解决评估器

此评估器判断客户的问题是否真正得到解决：


In [ ]:
resolution_evaluator = ClassificationEvaluator(
    name="resolution_status",
    prompt_template="""你正在评估客户的问题是否在客服对话中得到解决。

满足以下条件时为 RESOLVED：
- 客户获得所需信息
- 问题得到回答
- 对话结束时客户需求得到满足

以下情况为 UNRESOLVED：
- 客户未获得所需内容
- 问题未得到回答
- 智能体无法帮助处理请求

[完整对话]：
{input}

客户问题是否得到解决？""",
    llm=llm,
    choices={"resolved": 1, "unresolved": 0},
)

### 运行会话评估
从 Phoenix 获取 span，按会话 ID 分组，然后评估每个会话：

In [ ]:
from phoenix.client.resources.sessions import SessionAnnotationData

# 获取所有智能体跨度
spans = phoenix_client.spans.get_spans(
    project_identifier="support-bot",
    limit=200,
)

# 筛选智能体跨度，并按会话 ID 分组
agent_spans = [span for span in spans if span.get("name") == "support-agent"]

session_groups: Dict[str, List[Any]] = {}
for span in agent_spans:
    # 获取属性（可能为字典或 JSON 字符串）
    attributes = span.get("attributes", {})
    if isinstance(attributes, str):
        attributes = json.loads(attributes)

    session_id = attributes.get("session.id") or attributes.get(SpanAttributes.SESSION_ID)
    if session_id:
        if session_id not in session_groups:
            session_groups[session_id] = []
        session_groups[session_id].append(span)

print(f"已找到 {len(session_groups)} 个会话")

# 评估每个会话
session_annotations = []

for session_id, session_spans in session_groups.items():
    # 按对话轮次排序
    session_spans.sort(
        key=lambda s: (
            json.loads(s.get("attributes", "{}"))
            if isinstance(s.get("attributes"), str)
            else s.get("attributes", {})
        ).get("conversation.turn", 0)
    )

    # 构建对话记录
    transcript_parts = []
    for i, span in enumerate(session_spans):
        # 获取属性
        attributes = span.get("attributes", {})
        if isinstance(attributes, str):
            attributes = json.loads(attributes)

        input_value = attributes.get("input.value", "")
        output_value = attributes.get("output.value", "")
        turn_num = attributes.get("conversation.turn", i + 1)

        transcript_parts.append(f"Turn {turn_num}:\nUser: {input_value}\nAgent: {output_value}")

    transcript = "\n\n".join(transcript_parts)

    if not transcript.strip():
        continue

    coherence_result = conversation_coherence_evaluator.evaluate({"input": transcript})
    coherence_score = (
        coherence_result[0] if isinstance(coherence_result, list) else coherence_result
    )

    resolution_result = resolution_evaluator.evaluate({"input": transcript})
    resolution_score = (
        resolution_result[0] if isinstance(resolution_result, list) else resolution_result
    )

    session_annotations.append(
        SessionAnnotationData(
            session_id=session_id,
            name="conversation_coherence",
            annotator_kind="LLM",
            result={
                "label": coherence_score.label,
                "score": coherence_score.score
                if hasattr(coherence_score, "score")
                else (1.0 if coherence_score.label == "coherent" else 0.0),
            },
            metadata={"model": "gpt-5", "turnCount": len(session_spans)},
        )
    )

    session_annotations.append(
        SessionAnnotationData(
            session_id=session_id,
            name="resolution_status",
            annotator_kind="LLM",
            result={
                "label": resolution_score.label,
                "score": resolution_score.score
                if hasattr(resolution_score, "score")
                else (1.0 if resolution_score.label == "resolved" else 0.0),
            },
            metadata={"model": "gpt-5", "turnCount": len(session_spans)},
        )
    )

print(f"\n✅ 已评估 {len(session_groups)} 个会话")

### 记录会话标注

将所有会话级标注记录到 Phoenix：

In [ ]:
if session_annotations:
    print(f"\n📤 正在记录 {len(session_annotations)} 条会话标注到 Phoenix……")

    try:
        phoenix_client.sessions.log_session_annotations(
            session_annotations=session_annotations,
            sync=False,
        )
        print(f"✅ 已记录 {len(session_annotations)} 条会话标注")
        print(f"   - {len(session_annotations) // 2} 个会话已评估")
        print("   - 每个会话有 2 条标注：conversation_coherence 与 resolution_status")
    except Exception as error:
        print(f"❌ 记录会话标注失败： {error}")
else:
    print(
        "\n⚠️  没有可记录的会话标注。请先运行多轮对话。"
    )